# RustPower Lesson 5: DuckDB SQL Integration & Bi-directional Parquet Flow 🦆

In the previous lessons, we explored the high-level ECS grid API and low-level solver acceleration. This tutorial demonstrates the ultimate synergy between RustPower's ECS snapshot engine and Python's modern database ecosystem (DuckDB, Pandas, PyArrow).

We will implement a complete, bi-directional, memory-only data cycle:
1. **Solve & Export (`to_duckdb`)**: Run a power flow and serialize the full case configuration and output results to in-memory Parquet ZIP archives.
2. **Relational Analysis**: Use DuckDB to load the Parquet ZIP archives directly in-memory and join/query them using SQL.
3. **SQL/DataFrame Modification**: Modify the grid's load components (simulating a grid planning or optimization scenario).
4. **Re-ingestion (`from_duckdb`)**: Re-package the modified data and load it back into a new RustPower grid using the `load_parquet_case` API, then re-solve.

In [1]:
import rustpower
import pandas as pd
import numpy as np
import duckdb
import zipfile
import io
import os
import tempfile
import shutil

## 1. Solve and Export to Parquet

Let's load the IEEE 118 grid, solve it, and export both the grid parameters (the **Case**) and the simulation results (the **Output Results**). 

We use `grid.post_process()` to make sure the latest voltage results and branch loading levels are written back into the ECS world components before archiving.

In [2]:
grid = rustpower.PowerGrid(case_path='../cases/IEEE118/data.zip', branch_analysis=True)
grid.init_pf()
grid.solve()
grid.post_process()

# Export ZIP archives of Parquet files
case_zip = grid.get_parquet_case()
res_zip = grid.get_parquet_results()

print(f"Exported case ZIP size: {len(case_zip)/1024:.2f} KB")
print(f"Exported results ZIP size: {len(res_zip)/1024:.2f} KB")

## 2. Load into DuckDB for SQL Analysis

DuckDB is highly optimized for querying Parquet files. We can extract the ZIP files to a temporary directory and read them using DuckDB. 

Because Bevy ECS stores components in different *Archetypes* (which map to separate Parquet files), we use DuckDB's `union_by_name=True` option. This joins all different archetypes into a single consolidated view automatically.

In [3]:
tmp_dir = tempfile.mkdtemp()
case_dir = os.path.join(tmp_dir, 'case')
res_dir = os.path.join(tmp_dir, 'res')

# Extract case and results ZIP files to the temp directory
with zipfile.ZipFile(io.BytesIO(case_zip)) as z:
    z.extractall(case_dir)
with zipfile.ZipFile(io.BytesIO(res_zip)) as z:
    z.extractall(res_dir)

# Connect to DuckDB (in-memory)
con = duckdb.connect()

# Create views linking all Parquet files in the directories
con.execute(f"CREATE VIEW case_world AS SELECT * FROM read_parquet('{case_dir}/archetypes/*.parquet', union_by_name=True)")
con.execute(f"CREATE VIEW res_world AS SELECT * FROM read_parquet('{res_dir}/archetypes/*.parquet', union_by_name=True)")

# Run a SQL query to join the line parameters with the solved results using the ECS Entity ID
line_loading = con.execute("""
    SELECT 
        c.id as entity_id,
        c."FromBus.item" as from_bus,
        c."ToBus.item" as to_bus,
        c."LineParams.r_ohm_per_km" as resistance,
        r."LineResultData.p_from_mw" as p_mw,
        r."LineResultData.loading_percent" as loading
    FROM case_world c
    JOIN res_world r ON c.id = r.id
    WHERE c."LineParams.r_ohm_per_km" IS NOT NULL
    ORDER BY loading DESC
    LIMIT 5
""").df()

print("--- Top 5 Loaded Lines from DuckDB ---")
print(line_loading)

## 3. SQL-Based Grid Modification

In a real-world scenario, you might want to modify grid properties (e.g. increase load levels, take lines out of service) using SQL or Pandas, and run a new power flow.

Let's identify the archetype file containing the `LoadCfg` component, modify it by scaling all active power loads (`p_mw.item`) by 1.5x, and save the parquet file back.

In [4]:
# Find the parquet file containing the LoadCfg components
load_arch_file = None
for filename in os.listdir(os.path.join(case_dir, 'archetypes')):
    if filename.endswith('.parquet'):
        path = os.path.join(case_dir, 'archetypes', filename)
        df = pd.read_parquet(path)
        if 'p_mw.item' in df.columns and 'LoadCfg.load_type' in df.columns:
            load_arch_file = path
            break

print(f"Modifying archetype file: {os.path.basename(load_arch_file)}")

# Read the original load table
load_df = pd.read_parquet(load_arch_file)
print("Original first 5 loads:")
print(load_df[['id', 'p_mw.item', 'q_mvar.item']].dropna().head())

# Scale load active power values by 1.5x
load_df['p_mw.item'] = load_df['p_mw.item'] * 1.5

# Save it back to overwrite the Parquet file
load_df.to_parquet(load_arch_file)

# Package the directory back into a ZIP archive
new_case_zip_buffer = io.BytesIO()
with zipfile.ZipFile(new_case_zip_buffer, 'w', zipfile.ZIP_DEFLATED) as new_zip:
    for root, dirs, files in os.walk(case_dir):
        for file in files:
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(file_path, case_dir)
            new_zip.write(file_path, rel_path)

new_case_zip = new_case_zip_buffer.getvalue()
print(f"\nRe-packaged modified ZIP size: {len(new_case_zip)/1024:.2f} KB")

## 4. Ingest and Solve the Modified Case

We load the modified case ZIP archive back into a new `PowerGrid` instance using the `load_parquet_case()` API and re-run the power flow.

Because loads have increased by 50%, we expect voltages throughout the network to drop significantly.

In [5]:
# Initialize a new grid and load the modified case Parquet data
grid_modified = rustpower.PowerGrid()
grid_modified.load_parquet_case(new_case_zip)

# Solve the new grid state
report_modified = grid_modified.solve()
print(f"Solve converged: {report_modified.converged}")
print(f"Iterations:      {report_modified.iterations}")

# Compare average voltages
orig_v = np.abs(grid.v)
mod_v = np.abs(grid_modified.v)

print(f"Original average voltage: {np.mean(orig_v):.4f} p.u.")
print(f"Modified average voltage: {np.mean(mod_v):.4f} p.u.")
print(f"Maximum voltage drop:     {np.max(orig_v - mod_v):.4f} p.u.")

# Cleanup temp directories
shutil.rmtree(tmp_dir)